In [1]:
import wandb
from datetime import datetime, timezone

wandb.login()
api = wandb.Api()

# 1. Define your project path
project_path = "tidiane/patch_icl_eval"

# 2. Get today's starting timestamp in UTC (W&B uses UTC)
timestamp_str  = "2026-02-23T00:00:00Z"  # For testing purposes, you can hardcode a date

# 3. Fetch runs using a complex MongoDB-style filter
# This grabs runs that are newer than the timestamp AND match either the universeg or patch_icl conditions.
run_filters = {
    "$and": [
        {"created_at": {"$gt": timestamp_str}},
        {
            "$or": [
                # Condition 1: It's just a universeg run
                {"config.method": "universeg"},
                
                # Condition 2: It's a patch_icl run AND the input_size is not null
                {
                    "$and": [
                        {"config.method": "patch_icl"},
                        {"config.model.patch_icl.feature_extractor.input_size": {"$ne": None}}
                    ]
                }
            ]
        }
    ]
}

filtered_runs = api.runs(
    path=project_path,
    filters=run_filters
)

runs_data = {}

# 4. Iterate through the fetched runs and extract metrics
for run in filtered_runs:
    print(f"Processing run: {run.name} (ID: {run.id})")
    
    try:
        # Store in your dictionary
        method = run.config.get("method")
        if method == "universeg":
            input_size = run.config.get("model", {}).get("universeg", {}).get("input_size")
        elif method == "patch_icl":
            input_size = run.config.get("model", {}).get("patch_icl", {}).get("feature_extractor", {}).get("input_size")
        else:            
            input_size = None  

        runs_data[run.name] = {
            "wandb_id": run.id,
            "config": run.config,
            "method": method,
            "input_size": input_size,
            "context_size" : run.config.get("context_size"),
            "gflops_per_sample": run.summary.get("gflops_per_sample"), 
            "val_final_dice": run.summary.get("val_final_dice"), 
        }
    except Exception as e:
        print(f"  -> Error processing run {run.name}: {e}")

print(f"\nSuccessfully downloaded data for {len(runs_data)} runs.")

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Processing run: silver-dawn-103 (ID: azy5wotf)
Processing run: confused-mountain-104 (ID: effsv0dn)
Processing run: efficient-pine-107 (ID: uaqa50ae)
Processing run: daily-gorge-108 (ID: lefvdage)
Processing run: hardy-firebrand-109 (ID: ek7xolx4)
Processing run: volcanic-shadow-111 (ID: 5mqkd5pw)
Processing run: true-fog-114 (ID: ocn1skod)
Processing run: dauntless-firebrand-140 (ID: 52d9thnn)
Processing run: fallen-blaze-145 (ID: si9rr1ui)
Processing run: clear-haze-147 (ID: wc7o83tk)

Successfully downloaded data for 10 runs.


In [2]:
for run_name, data in runs_data.items():
    #print(f"\nRun: {run_name}")
    #print(f"W&B ID: {data['wandb_id']}")
    print(f"Method: {data['method']}")
    print(f"context_size: {data['context_size']}")
    print(f"input_size: {data['input_size']}")
    print(f"gflops_per_sample: {data['gflops_per_sample']}")
    print(f"val_final_dice: {data['val_final_dice']}")
    #print("Per-case Dice DataFrame:")
    #print(data["per_case_dice"].head())

Method: universeg
context_size: 1
input_size: 128
gflops_per_sample: 13.572767744
val_final_dice: 0.07844752718624198
Method: universeg
context_size: 3
input_size: 128
gflops_per_sample: 34.334572544
val_final_dice: 0.18635937626592616
Method: universeg
context_size: 15
input_size: 128
gflops_per_sample: 158.905401344
val_final_dice: 0.3481535982127452
Method: universeg
context_size: 10
input_size: 64
gflops_per_sample: 26.750222336
val_final_dice: 0.25033283267104117
Method: universeg
context_size: 10
input_size: 128
gflops_per_sample: 107.000889344
val_final_dice: 0.31396038318453745
Method: universeg
context_size: 10
input_size: 256
gflops_per_sample: 428.003557376
val_final_dice: 0.2908167294533067
Method: universeg
context_size: 10
input_size: 512
gflops_per_sample: 1712.014229504
val_final_dice: 0.22743874287015903
Method: patch_icl
context_size: 10
input_size: 256
gflops_per_sample: 204.364732416
val_final_dice: 0.15071565924315078
Method: patch_icl
context_size: 10
input_size: 

In [3]:
import matplotlib.pyplot as plt
import pandas as pd

# Set publication-ready style
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.autolayout': True,
    'axes.linewidth': 1.5,
})

# 1. Convert your 'runs_data' dictionary into a Pandas DataFrame
# (Assuming runs_data is still a dictionary of dictionaries from your previous code)
df = pd.DataFrame.from_dict(runs_data, orient='index')

def plot_pareto(df, vary_col, fixed_col, fixed_val, title, filename, label_prefix):
    fig, ax = plt.subplots(figsize=(7, 5))
    
    # 2. Filter for the specific experiment
    # For example, filter where input_size == 128 when studying context_size
    plot_df = df[df[fixed_col] == fixed_val].copy()
    
    # Optional: ensure we actually have data to plot
    if plot_df.empty:
        print(f"No data found for {fixed_col} == {fixed_val}. Skipping plot.")
        return

    methods = plot_df['method'].unique()
    colors = {'universeg': 'tab:blue', 'patch_icl': 'tab:orange'}
    markers = {'universeg': 'o', 'patch_icl': 's'}
    
    # 3. Plot each method as its own line
    for method in methods:
        # Crucial step: Sort by GFLOPs so the line traces the Pareto front correctly!
        method_df = plot_df[plot_df['method'] == method].sort_values('gflops_per_sample')
        
        ax.plot(method_df['gflops_per_sample'], method_df['val_final_dice'], 
                marker=markers.get(method, 'o'), color=colors.get(method, 'k'), 
                label=method, markersize=8, linewidth=2.5, alpha=0.8)
        
        # 4. Annotate each data point with its context_size or input_size value
        for _, row in method_df.iterrows():
            ax.annotate(f"{label_prefix}={int(row[vary_col])}", 
                        (row['gflops_per_sample'], row['val_final_dice']),
                        textcoords="offset points", 
                        xytext=(0, 10),  # Offset label slightly above the point
                        ha='center', 
                        fontsize=10,
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.7))

    # Add labels and formatting
    ax.set_xlabel('Computational Cost (GFLOPs / Sample)', fontweight='bold')
    ax.set_ylabel('Performance (Validation Dice)', fontweight='bold')
    ax.set_title(title, pad=15, fontweight='bold')
    ax.legend(title="Method")
    ax.grid(True, linestyle=':', alpha=0.7)
    
    # 5. Handle massive scale differences using a log scale
    # Since Input Size GFLOPs jump from ~20 to ~1700, a log scale is visually better
    if vary_col == 'input_size':
        ax.set_xscale('log')
        ax.set_xlabel('Computational Cost (GFLOPs / Sample) - Log Scale', fontweight='bold')
    
    # Save the figure
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

# --- Execution ---

# 1. Plot for Varying Context Size (Fixing Input Size to 128)
plot_pareto(df, 
            vary_col='context_size', 
            fixed_col='input_size', 
            fixed_val=128, 
            title='Compute vs Performance: Varying Context Size\n(Fixed Input Size: 128)', 
            filename='pareto_context_size.pdf', # PDF is great for LaTeX/Overleaf
            label_prefix='ctx')

# 2. Plot for Varying Input Size (Fixing Context Size to 10)
plot_pareto(df, 
            vary_col='input_size', 
            fixed_col='context_size', 
            fixed_val=10, 
            title='Compute vs Performance: Varying Input Size\n(Fixed Context Size: 10)', 
            filename='pareto_input_size.pdf',
            label_prefix='in')